In [ ]:
import numpy as np
import pandas as pd
from GTZANClassifier import GTZANClassifier
from sklearn.metrics import balanced_accuracy_score

In [2]:
sub_folder = '/og_params' 

feat_types = ['full', 'mfcc'] # , 'spectral'
rng_states = ['42', '351', '605'] # , '351', '605'
time_states = ['30'] # '3', 

classifers = {}
run_meta = {}

for t in time_states:
    for feat in feat_types:
        for rng in rng_states:
            name = f"{t}s_{feat}_rng{rng}"
            clf = GTZANClassifier()
            clf.load_models(
                pkl_filepath=f"./trained_models{sub_folder}/{name}.pkl",
                data_dirpath=f"./GTZAN/Data_filtered/{t}s_split/",
            )
            classifers[name] = clf
            run_meta[name] = {
                "name": name,
                "duration": f"{t}s",
                "feat_type": feat,
                "rng": int(rng),
            }

Loading TRAIN split from: train_filtered_30s_features.csv
Dataset 'GTZAN\Data_filtered\30s_split\train_filtered_30s_features.csv' has 0 missing entries.
Feature Subset: Full feature set.
Loading TEST split from: test_filtered_30s_features.csv
Dataset 'GTZAN\Data_filtered\30s_split\test_filtered_30s_features.csv' has 0 missing entries.
Feature Subset: Full feature set.
Loading VAL split from: valid_filtered_30s_features.csv
Dataset 'GTZAN\Data_filtered\30s_split\valid_filtered_30s_features.csv' has 0 missing entries.
Feature Subset: Full feature set.
Train size: 463 | Val size: 203 | Test size: 300
-> Data splits successfully reconstructed for '30s_full_rng42'.
Models successfully loaded from './trained_models/og_params/30s_full_rng42.pkl'.

Loading TRAIN split from: train_filtered_30s_features.csv
Dataset 'GTZAN\Data_filtered\30s_split\train_filtered_30s_features.csv' has 0 missing entries.
Feature Subset: Full feature set.
Loading TEST split from: test_filtered_30s_features.csv
Datase

In [3]:
def extract_cv_df(bundle, meta, metric="mean_test_score"):
    cv = bundle.rf_results.cv_results_
    std_key = f"std_{metric.replace('mean_', '')}"

    raw_depths = [p["rf__max_depth"] for p in cv["params"]]
    clean_depths = [None if pd.isna(d) else int(d) for d in raw_depths]

    df = pd.DataFrame({
        "classifier_key": meta["name"],
        "duration": meta["duration"],
        "feat_type": meta["feat_type"],
        "rng": meta["rng"],
        "max_depth": clean_depths,
        "n_estimators": [int(p["rf__n_estimators"]) for p in cv["params"]],
        "min_samples_leaf": [int(p["rf__min_samples_leaf"]) for p in cv["params"]],
        "cv_score": cv[metric],
        "cv_std": cv.get(std_key, np.nan),
    })
    df["depth_label"] = df["max_depth"].apply(
        lambda d: "None (unbounded)" if pd.isna(d) else f"{int(d):02d}"
    )
    return df

def build_grid_dataframe(classifers_dict, meta_dict, metric="mean_test_score"):
    frames = [
        extract_cv_df(classifers_dict[k], meta_dict[k], metric=metric)
        for k in classifers_dict
        if k in meta_dict
    ]
    return pd.concat(frames, ignore_index=True)



In [4]:
def compute_paired_lift(grid_df, baseline_feat="mfcc", target_feat="full"):
    join_cols = ["duration", "rng", "max_depth", "depth_label", "n_estimators", "min_samples_leaf"]

    df_target = (
        grid_df[grid_df["feat_type"] == target_feat][join_cols + ["cv_score"]]
        .rename(columns={"cv_score": f"score_{target_feat}"})
    )
    df_base = (
        grid_df[grid_df["feat_type"] == baseline_feat][join_cols + ["cv_score"]]
        .rename(columns={"cv_score": f"score_{baseline_feat}"})
    )

    paired = pd.merge(df_target, df_base, on=join_cols, how="inner")
    paired["lift"] = paired[f"score_{target_feat}"] - paired[f"score_{baseline_feat}"]
    return paired



In [5]:
def _best_params_per_depth(gs):
    cv = gs.cv_results_
    df = pd.DataFrame({
        "mean_cv": cv["mean_test_score"],
        "max_depth": [p["rf__max_depth"] for p in cv["params"]],
        "n_estimators": [p["rf__n_estimators"] for p in cv["params"]],
        "min_samples_leaf": [p["rf__min_samples_leaf"] for p in cv["params"]],
    })
    return (
        df.sort_values("mean_cv", ascending=False)
        .groupby("max_depth", dropna=False, as_index=False)
        .first()
    )

def _get_train_xy(bundle, key_tuple, train_splits=None):
    if train_splits and key_tuple in train_splits:
        return train_splits[key_tuple]
    for x_attr, y_attr in [("X_train", "y_train"), ("x_train", "y_train")]:
        if hasattr(bundle, x_attr) and hasattr(bundle, y_attr):
            return getattr(bundle, x_attr), getattr(bundle, y_attr)
    raise AttributeError(
        f"Training data not found on bundle for {key_tuple}; pass train_splits={{(duration, feat, rng): (X_train, y_train)}}."
    )

def eval_heldout_by_depth(
    classifers_dict,
    meta_dict,
    depths=(5, 10, 20, None),
    baseline_feat="mfcc",
    target_feat="full",
    train_splits=None,
    scorer_fn=balanced_accuracy_score,
):
    by_key = {
        (m["duration"], m["rng"], m["feat_type"]): (k, classifers_dict[k])
        for k, m in meta_dict.items()
        if k in classifers_dict
    }
    combos = sorted({(d, r) for (d, r, _) in by_key})
    rows = []

    for duration, rng in combos:
        item_base = by_key.get((duration, rng, baseline_feat))
        item_target = by_key.get((duration, rng, target_feat))
        if item_base is None or item_target is None:
            continue

        _, b_base = item_base
        _, b_target = item_target

        X_tr_b, y_tr_b = _get_train_xy(b_base, (duration, baseline_feat, rng), train_splits)
        X_tr_t, y_tr_t = _get_train_xy(b_target, (duration, target_feat, rng), train_splits)

        best_b = _best_params_per_depth(b_base.rf_results)
        best_t = _best_params_per_depth(b_target.rf_results)

        for depth in depths:
            mask_b = best_b["max_depth"].isna() if depth is None else (best_b["max_depth"] == depth)
            mask_t = best_t["max_depth"].isna() if depth is None else (best_t["max_depth"] == depth)
            if not mask_b.any() or not mask_t.any():
                continue

            r_b = best_b[mask_b].iloc[0]
            r_t = best_t[mask_t].iloc[0]

            est_b = b_base.rf_results.estimator.set_params(
                rf__max_depth=depth,
                rf__n_estimators=int(r_b["n_estimators"]),
                rf__min_samples_leaf=int(r_b["min_samples_leaf"]),
            ).fit(X_tr_b, y_tr_b)

            est_t = b_target.rf_results.estimator.set_params(
                rf__max_depth=depth,
                rf__n_estimators=int(r_t["n_estimators"]),
                rf__min_samples_leaf=int(r_t["min_samples_leaf"]),
            ).fit(X_tr_t, y_tr_t)

            val_b = scorer_fn(b_base.y_val, est_b.predict(b_base.X_val))
            val_t = scorer_fn(b_target.y_val, est_t.predict(b_target.X_val))

            rows.append({
                "duration": duration,
                "rng": rng,
                "max_depth": "None" if depth is None else depth,
                "val_lift": val_t - val_b,
                f"val_{target_feat}": val_t,
                f"val_{baseline_feat}": val_b,
            })

    heldout_df = pd.DataFrame(rows)
    heldout_summary = (
        heldout_df.groupby(["duration", "max_depth"], dropna=False)
        .agg(
            n_seeds=("rng", "nunique"),
            mean_val_lift=("val_lift", "mean"),
            std_val_lift=("val_lift", "std"),
        )
    )
    return heldout_df, heldout_summary

In [6]:
grid_df = build_grid_dataframe(classifers, run_meta)

paired = compute_paired_lift(grid_df, baseline_feat="mfcc", target_feat="full")

depth_summary = (
    paired.groupby(["duration", "depth_label", "max_depth"], dropna=False)
    .agg(
        n_pairs=("lift", "count"),
        n_seeds=("rng", "nunique"),
        mean_mfcc=("score_mfcc", "mean"),
        mean_full=("score_full", "mean"),
        mean_lift=("lift", "mean"),
        median_lift=("lift", "median"),
        std_lift=("lift", "std"),
    )
    .sort_values(["duration", "max_depth"], key=lambda s: s.fillna(np.inf) if s.name == "max_depth" else s)
)

rng_by_depth = (
    paired.groupby(["duration", "rng", "depth_label", "max_depth"], dropna=False)["lift"]
    .mean()
    .unstack(["depth_label", "max_depth"])
)

In [7]:
print("=== Mean CV Lift by max_depth (across all seeds & grid settings) ===")
print(depth_summary.round(4))

print("\n=== Mean CV Lift by Random State (rng) and max_depth ===")
print(rng_by_depth.round(4))

heldout_df, heldout_summary = eval_heldout_by_depth(
    classifers_dict=classifers,
    meta_dict=run_meta,
    depths=(5, 10, 20, None),
    baseline_feat="mfcc",
    target_feat="full",
    scorer_fn=balanced_accuracy_score,
)

print("\n=== Held-out Validation Lift Summary by max_depth ===")
print(heldout_summary.round(4))

print("\n=== Per-Seed Held-out Breakdown ===")
print(heldout_df.round(4))

=== Mean CV Lift by max_depth (across all seeds & grid settings) ===
                                     n_pairs  n_seeds  mean_mfcc  mean_full  \
duration depth_label      max_depth                                           
30s      05               5.0             27        3     0.6565     0.6764   
         10               10.0            27        3     0.6696     0.7087   
         20               20.0            27        3     0.6700     0.7078   
         None (unbounded) NaN             27        3     0.6700     0.7078   

                                     mean_lift  median_lift  std_lift  
duration depth_label      max_depth                                    
30s      05               5.0           0.0199       0.0196    0.0108  
         10               10.0          0.0392       0.0399    0.0188  
         20               20.0          0.0378       0.0400    0.0163  
         None (unbounded) NaN           0.0378       0.0400    0.0163  

=== Mean CV Lift by Ran